## Training of the valuation of CAT bonds - Full cross-validation

In [2]:
import os
# Suppress TensorFlow logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adam

import time
import itertools
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --------------------------------------------------
# Set random seed for reproducibility
# --------------------------------------------------
SEED = 125

# Python
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# TensorFlow/Keras
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic TensorFlow operations (TF >= 2.9)
tf.config.experimental.enable_op_determinism()

print(f"Random seed set to {SEED}")

Random seed set to 125


## Model-Building Function
Create a function that builds a Keras model based on input parameters. This function allows flexibility in specifying the number of layers, neurons, activation functions, regularization, etc.

In [4]:
def build_model(input_dim,
                nr_neurons,
                reg_param=0.00,
                activation='relu',
                use_batch_norm=True,
                dropout_rate=0.0,
                learning_rate=0.005):
    """
    Builds a customizable neural network model for regression tasks.

    Parameters:
    - input_dim (int): Number of input features.
    - nr_neurons (list): List defining the number of neurons in each hidden layer.
                            Example: [64, 32] for 2 layers with 64 and 32 neurons.
    - reg_param (float): L2 regularization parameter.
    - activation (str): Activation function for hidden layers.
    - use_batch_norm (bool): Whether to use Batch Normalization.
    - dropout_rate (float): Dropout rate for regularization.

    Returns:
    - model (keras.Model): Compiled Keras model.
    """
    model = models.Sequential()

    # Input layer (first hidden layer)
    model.add(layers.Dense(nr_neurons[0],
                           kernel_regularizer=regularizers.l2(reg_param),
                           activation=activation,
                           input_shape=(input_dim,)))

    if use_batch_norm:
        model.add(layers.BatchNormalization())
    if dropout_rate > 0.0:
        model.add(layers.Dropout(dropout_rate))

    # Add remaining hidden layers
    for neurons in nr_neurons[1:]:
        model.add(layers.Dense(neurons,
                               kernel_regularizer=regularizers.l2(reg_param),
                               activation=activation))
        if use_batch_norm:
            model.add(layers.BatchNormalization())
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))

    # Output layer
    model.add(layers.Dense(1))  # Single output for regression

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse',
                  metrics=['mae', 'mse'])

    return model

In [5]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

## Define Configurations to Test
Specify the different configurations you want to test. For example, varying the number of neurons, activation functions, regularization parameters, etc.

In [21]:
# Define different configurations
layer_configs = [
    [128, 64, 32],        # 3 layers: 128 -> 64 -> 32 neurons
    [256, 128, 64, 32]       # 4 layers: 256 -> 128 -> 64 -> 32 neurons
]

activations = ['relu']
reg_params = [1e-04]
dropout_rates = [0.1]
use_batch_norm_options = [True]
learning_rates = [1e-04, 1e-05]

# Create all possible combinations
configurations = list(itertools.product(layer_configs, activations, reg_params, dropout_rates, use_batch_norm_options, learning_rates))

print(f"Total configurations to test: {len(configurations)}")

Total configurations to test: 4


## Read the data - Gamma Distribution

In [24]:
# Load your dataset (replace with your actual file path)
df = pd.read_csv("CAT_price_log.csv")

print(df.head())
print(df.shape)

      c         r  kappa  theta  sigma     lambda             D   N         T  \
0  0.05  0.047464    0.2   0.03   0.02  37.595368  1.067337e+10  10  1.971323   
1  0.05  0.005287    0.2   0.03   0.02  30.240079  1.244350e+10  10  1.947998   
2  0.05  0.037334    0.2   0.03   0.02  33.820770  9.731976e+09  10  1.557770   
3  0.05  0.026329    0.2   0.03   0.02  37.278229  1.171317e+10   8  0.815643   
4  0.05  0.046435    0.2   0.03   0.02  31.427952  1.049263e+10  12  0.832221   

         Price  
0   680.036064  
1  1384.209831  
2  1179.769620  
3  1373.370457  
4  1548.602842  
(600000, 10)


In [26]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price'] / 1000

print(X.head())
print(y.head())

          r     lambda             D   N         T
0  0.047464  37.595368  1.067337e+10  10  1.971323
1  0.005287  30.240079  1.244350e+10  10  1.947998
2  0.037334  33.820770  9.731976e+09  10  1.557770
3  0.026329  37.278229  1.171317e+10   8  0.815643
4  0.046435  31.427952  1.049263e+10  12  0.832221
0    0.680036
1    1.384210
2    1.179770
3    1.373370
4    1.548603
Name: Price, dtype: float64


## Implement K-Fold Cross-Validation
Use K-Fold Cross-Validation to evaluate each configuration robustly.

In [29]:
# Parameters
EPOCHS = 500
BATCH_SIZE = 512
NR_FOLDS = 5  # Number of folds for cross-validation

# Initialize K-Fold
kf = KFold(n_splits=NR_FOLDS, shuffle=True, random_state=125)

# Prepare a DataFrame to store results
results = []

# Measure training time
start_time = time.perf_counter()

# Iterate over each configuration
for idx, (layer_config, activation, reg_param, dropout_rate, use_batch_norm, learning_rate) in enumerate(configurations):
    print(f"\n--- Configuration {idx+1}/{len(configurations)} ---")
    print(f"Layers: {layer_config}, Activation: {activation}, L2: {reg_param}, Dropout: {dropout_rate}, BatchNorm: {use_batch_norm}, LearningRate: {learning_rate}")

    # Lists to store metrics for each fold
    fold_losses = []
    fold_maes = []
    fold_mses = []

    # Iterate over each fold
    for fold, (train_index, val_index) in enumerate(kf.split(X)):
        print(f"  Fold {fold+1}/{NR_FOLDS}")

        # Split data into training and validation for this fold
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Scale the data
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        # Build the model
        model = build_model(input_dim=X_train_fold_scaled.shape[1],
                            nr_neurons=layer_config,
                            reg_param=reg_param,
                            activation=activation,
                            use_batch_norm=use_batch_norm,
                            dropout_rate=dropout_rate,
                            learning_rate=learning_rate)
        
#         model.summary()

        # Train the model
        history = model.fit(X_train_fold_scaled, y_train_fold,
                            epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            validation_data=(X_val_fold_scaled, y_val_fold),
                            callbacks=[early_stop],
                            verbose=0)  # Set verbose=1 for detailed output

        # Evaluate the model
        val_loss, val_mae, val_mse = model.evaluate(X_val_fold_scaled, y_val_fold, verbose=0)
        print(f"    Loss: {val_loss:.4f}, MAE: {val_mae:.4f}, MSE: {val_mse:.4f}")

        # Append metrics
        fold_losses.append(val_loss)
        fold_maes.append(val_mae)
        fold_mses.append(val_mse)

    # Calculate average metrics across folds
    avg_loss = np.mean(fold_losses)
    avg_mae = np.mean(fold_maes)
    avg_mse = np.mean(fold_mses)

    print(f"  Average Loss: {avg_loss:.4f}, Average MAE: {avg_mae:.4f}, Average MSE: {avg_mse:.4f}")

    # Store the results
    results.append({
        'Layers': layer_config,
        'Activation': activation,
        'L2_Regularization': reg_param,
        'Dropout_Rate': dropout_rate,
        'Batch_Normalization': use_batch_norm,
        'Learning_Rate': learning_rate,
        'Average_Loss': avg_loss,
        'Average_MAE': avg_mae,
        'Average_MSE': avg_mse
    })

    # Optional: Clear the Keras session to free memory after each configuration
    keras.backend.clear_session()


training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")


--- Configuration 1/4 ---
Layers: [128, 64, 32], Activation: relu, L2: 0.0001, Dropout: 0.1, BatchNorm: True, LearningRate: 0.0001
  Fold 1/5
    Loss: 0.0001, MAE: 0.0047, MSE: 0.0000
  Fold 2/5
    Loss: 0.0001, MAE: 0.0071, MSE: 0.0001
  Fold 3/5
    Loss: 0.0001, MAE: 0.0045, MSE: 0.0000
  Fold 4/5
    Loss: 0.0001, MAE: 0.0048, MSE: 0.0000
  Fold 5/5
    Loss: 0.0001, MAE: 0.0053, MSE: 0.0000
  Average Loss: 0.0001, Average MAE: 0.0053, Average MSE: 0.0000

--- Configuration 2/4 ---
Layers: [128, 64, 32], Activation: relu, L2: 0.0001, Dropout: 0.1, BatchNorm: True, LearningRate: 1e-05
  Fold 1/5
    Loss: 0.0001, MAE: 0.0039, MSE: 0.0000
  Fold 2/5
    Loss: 0.0001, MAE: 0.0052, MSE: 0.0001
  Fold 3/5
    Loss: 0.0001, MAE: 0.0043, MSE: 0.0000
  Fold 4/5
    Loss: 0.0001, MAE: 0.0045, MSE: 0.0000
  Fold 5/5
    Loss: 0.0001, MAE: 0.0035, MSE: 0.0000
  Average Loss: 0.0001, Average MAE: 0.0043, Average MSE: 0.0000

--- Configuration 3/4 ---
Layers: [256, 128, 64, 32], Activation: 

## Analyze and Visualize Results
After testing all configurations, convert the results into a pandas DataFrame for easier analysis and visualization.

In [31]:
# Convert results to DataFrame
results_gamma = pd.DataFrame(results)

# Display the results
print("\n--- Cross-Validation Results ---")
print(results_gamma.sort_values(by='Average_MSE').reset_index(drop=True))


--- Cross-Validation Results ---
               Layers Activation  L2_Regularization  Dropout_Rate  \
0  [256, 128, 64, 32]       relu             0.0001           0.1   
1       [128, 64, 32]       relu             0.0001           0.1   
2       [128, 64, 32]       relu             0.0001           0.1   
3  [256, 128, 64, 32]       relu             0.0001           0.1   

   Batch_Normalization  Learning_Rate  Average_Loss  Average_MAE  Average_MSE  
0                 True        0.00001      0.000076     0.003503     0.000023  
1                 True        0.00001      0.000090     0.004268     0.000034  
2                 True        0.00010      0.000105     0.005269     0.000048  
3                 True        0.00010      0.000141     0.006047     0.000071  


## Selecting the Best Configuration
Based on the results, you can select the configuration with the lowest average MSE or any other metric that suits your needs.

In [33]:
# Find the best configuration based on Average MSE
best_config_gamma = results_gamma.loc[results_gamma['Average_MSE'].idxmin()]
print("\n--- Best Configuration ---")
print(best_config_gamma)


--- Best Configuration ---
Layers                 [256, 128, 64, 32]
Activation                           relu
L2_Regularization                  0.0001
Dropout_Rate                          0.1
Batch_Normalization                  True
Learning_Rate                     0.00001
Average_Loss                     0.000076
Average_MAE                      0.003503
Average_MSE                      0.000023
Name: 3, dtype: object


## Read the data - Lognormal Distribution

In [35]:
# Load your dataset
df = pd.read_csv("CAT_price_log.csv")

print(df.head())
print(df.shape)

      c         r  kappa  theta  sigma     lambda             D   N         T  \
0  0.05  0.047464    0.2   0.03   0.02  37.595368  1.067337e+10  10  1.971323   
1  0.05  0.005287    0.2   0.03   0.02  30.240079  1.244350e+10  10  1.947998   
2  0.05  0.037334    0.2   0.03   0.02  33.820770  9.731976e+09  10  1.557770   
3  0.05  0.026329    0.2   0.03   0.02  37.278229  1.171317e+10   8  0.815643   
4  0.05  0.046435    0.2   0.03   0.02  31.427952  1.049263e+10  12  0.832221   

         Price  
0   680.036064  
1  1384.209831  
2  1179.769620  
3  1373.370457  
4  1548.602842  
(600000, 10)


In [36]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price'] / 1000

print(X.head())
print(y.head())

          r     lambda             D   N         T
0  0.047464  37.595368  1.067337e+10  10  1.971323
1  0.005287  30.240079  1.244350e+10  10  1.947998
2  0.037334  33.820770  9.731976e+09  10  1.557770
3  0.026329  37.278229  1.171317e+10   8  0.815643
4  0.046435  31.427952  1.049263e+10  12  0.832221
0    0.680036
1    1.384210
2    1.179770
3    1.373370
4    1.548603
Name: Price, dtype: float64


In [37]:
# Parameters
EPOCHS = 1000
BATCH_SIZE = 512
NR_FOLDS = 5  # Number of folds for cross-validation

# Initialize K-Fold
kf = KFold(n_splits=NR_FOLDS, shuffle=True, random_state=125)

# Prepare a DataFrame to store results
results = []

# Measure training time
start_time = time.perf_counter()

# Iterate over each configuration
for idx, (layer_config, activation, reg_param, dropout_rate, use_batch_norm, learning_rate) in enumerate(configurations):
    print(f"\n--- Configuration {idx+1}/{len(configurations)} ---")
    print(f"Layers: {layer_config}, Activation: {activation}, L2: {reg_param}, Dropout: {dropout_rate}, BatchNorm: {use_batch_norm}, LearningRate: {learning_rate}")

    # Lists to store metrics for each fold
    fold_losses = []
    fold_maes = []
    fold_mses = []

    # Iterate over each fold
    for fold, (train_index, val_index) in enumerate(kf.split(X)):
        print(f"  Fold {fold+1}/{NR_FOLDS}")

        # Split data into training and validation for this fold
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Scale the data
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        # Build the model
        model = build_model(input_dim=X_train_fold_scaled.shape[1],
                            nr_neurons=layer_config,
                            reg_param=reg_param,
                            activation=activation,
                            use_batch_norm=use_batch_norm,
                            dropout_rate=dropout_rate,
                            learning_rate=learning_rate)
        
#         model.summary()

        # Train the model
        history = model.fit(X_train_fold_scaled, y_train_fold,
                            epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            validation_data=(X_val_fold_scaled, y_val_fold),
                            callbacks=[early_stop],
                            verbose=0)  # Set verbose=1 for detailed output

        # Evaluate the model
        val_loss, val_mae, val_mse = model.evaluate(X_val_fold_scaled, y_val_fold, verbose=0)
        print(f"    Loss: {val_loss:.4f}, MAE: {val_mae:.4f}, MSE: {val_mse:.4f}")

        # Append metrics
        fold_losses.append(val_loss)
        fold_maes.append(val_mae)
        fold_mses.append(val_mse)

    # Calculate average metrics across folds
    avg_loss = np.mean(fold_losses)
    avg_mae = np.mean(fold_maes)
    avg_mse = np.mean(fold_mses)

    print(f"  Average Loss: {avg_loss:.4f}, Average MAE: {avg_mae:.4f}, Average MSE: {avg_mse:.4f}")

    # Store the results
    results.append({
        'Layers': layer_config,
        'Activation': activation,
        'L2_Regularization': reg_param,
        'Dropout_Rate': dropout_rate,
        'Batch_Normalization': use_batch_norm,
        'Learning_Rate': learning_rate,
        'Average_Loss': avg_loss,
        'Average_MAE': avg_mae,
        'Average_MSE': avg_mse
    })

    # Optional: Clear the Keras session to free memory after each configuration
    keras.backend.clear_session()


training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")


--- Configuration 1/4 ---
Layers: [128, 64, 32], Activation: relu, L2: 0.0001, Dropout: 0.1, BatchNorm: True, LearningRate: 0.0001
  Fold 1/5
    Loss: 0.0001, MAE: 0.0048, MSE: 0.0000
  Fold 2/5
    Loss: 0.0001, MAE: 0.0050, MSE: 0.0000
  Fold 3/5
    Loss: 0.0001, MAE: 0.0044, MSE: 0.0000
  Fold 4/5
    Loss: 0.0001, MAE: 0.0047, MSE: 0.0000
  Fold 5/5
    Loss: 0.0001, MAE: 0.0056, MSE: 0.0001
  Average Loss: 0.0001, Average MAE: 0.0049, Average MSE: 0.0000

--- Configuration 2/4 ---
Layers: [128, 64, 32], Activation: relu, L2: 0.0001, Dropout: 0.1, BatchNorm: True, LearningRate: 1e-05
  Fold 1/5
    Loss: 0.0001, MAE: 0.0039, MSE: 0.0000
  Fold 2/5
    Loss: 0.0001, MAE: 0.0041, MSE: 0.0000
  Fold 3/5
    Loss: 0.0001, MAE: 0.0044, MSE: 0.0000
  Fold 4/5
    Loss: 0.0001, MAE: 0.0041, MSE: 0.0000
  Fold 5/5
    Loss: 0.0001, MAE: 0.0040, MSE: 0.0000
  Average Loss: 0.0001, Average MAE: 0.0041, Average MSE: 0.0000

--- Configuration 3/4 ---
Layers: [256, 128, 64, 32], Activation: 

In [38]:
# Convert results to DataFrame
results_log = pd.DataFrame(results)

# Display the results
print("\n--- Cross-Validation Results ---")
print(results_log.sort_values(by='Average_MSE').reset_index(drop=True))


--- Cross-Validation Results ---
               Layers Activation  L2_Regularization  Dropout_Rate  \
0  [256, 128, 64, 32]       relu             0.0001           0.1   
1       [128, 64, 32]       relu             0.0001           0.1   
2       [128, 64, 32]       relu             0.0001           0.1   
3  [256, 128, 64, 32]       relu             0.0001           0.1   

   Batch_Normalization  Learning_Rate  Average_Loss  Average_MAE  Average_MSE  
0                 True        0.00001      0.000073     0.003266     0.000021  
1                 True        0.00001      0.000089     0.004125     0.000030  
2                 True        0.00010      0.000098     0.004898     0.000043  
3                 True        0.00010      0.000126     0.005564     0.000059  


In [39]:
# Find the best configuration based on Average MSE
best_config_log = results_log.loc[results_log['Average_MSE'].idxmin()]
print("\n--- Best Configuration ---")
print(best_config_log)


--- Best Configuration ---
Layers                 [256, 128, 64, 32]
Activation                           relu
L2_Regularization                  0.0001
Dropout_Rate                          0.1
Batch_Normalization                  True
Learning_Rate                     0.00001
Average_Loss                     0.000073
Average_MAE                      0.003266
Average_MSE                      0.000021
Name: 3, dtype: object
